# Bronze Layer — Source Blob reports/ → Bronze Volume (Monthly JSON Reports)

**Day 6 | Manual — partitioned by month YYYY/MM/**

Copies monthly JSON report files from the `reports/` folder in the instructor source blob into the Bronze Volume, preserving the `YYYY/MM/` partition structure.

---

### Source layout

```
wasbs://source@dataenggdailystorage.blob.core.windows.net/
  └── reports/
        └── YYYY/
              └── MM/
                    └── kpi_report_YYYYMM.json
                    └── sla_report_YYYYMM.json
                    └── state_breakdown_YYYYMM.json
```

Current data: **June 2026** only, 3 report files.

| File | Content |
|---|---|
| `kpi_report_202606.json` | KPI metrics (uptime, utilisation, revenue, sessions) |
| `sla_report_202606.json` | SLA metrics (response times, breach counts, availability) |
| `state_breakdown_202606.json` | State-level aggregations (sessions, revenue, faults per state) |

### Bronze Volume target

```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
  └── reports/
        └── YYYY/MM/
              └── *.json   (3 files per month)
```

---

### Load modes — controlled by `LOAD_MODE` in Cell 2

| `LOAD_MODE` | What it copies |
|---|---|
| `full` | All report months — use for first run |
| `monthly` | Only the specific `YYYY/MM` month folder — use for new month loads |

## Cell 1 — Authenticate to source blob storage

In [ ]:
SCOPE = "kv-ev-scope"

STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)

SOURCE_ROOT = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"Source root     : {SOURCE_ROOT}")
print("Source blob authenticated — OK")

## Cell 2 — Read `load_type` Job parameter and set paths

`load_type` is passed by the Databricks Job as a widget parameter.

| `load_type` | Behaviour |
|---|---|
| `full` | Copies all report months — use for first run |
| `incremental` | Copies only last month's folder (auto-computed from `datetime.now(UTC)`) |

When running interactively, the widget falls back to `"incremental"` if not set.

In [ ]:
from datetime import datetime, timezone
from dateutil.relativedelta import relativedelta   # pre-installed on Databricks Runtime

# Read load_type from Databricks Job widget parameter.
# Default: "incremental" so the notebook is safe to run manually without setting the widget.
dbutils.widgets.text("load_type", "incremental", "Load Type (full / incremental)")
load_type = dbutils.widgets.get("load_type").strip().lower()

if load_type not in ("full", "incremental"):
    raise ValueError(f"Invalid load_type='{load_type}'. Must be 'full' or 'incremental'.")

print(f"load_type : {load_type}")

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
BASE_SUBPATH  = "reports"

now = datetime.now(timezone.utc)

if load_type == "full":
    SOURCE_PATH = f"{SOURCE_ROOT}/{BASE_SUBPATH}/"
    BRONZE_PATH = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/"
    partition   = None
    print(f"Mode : full — all report months")
else:
    # Incremental: target last month (job fires on 1st of month at 02:00 UTC)
    last_month  = now - relativedelta(months=1)
    partition   = last_month.strftime("%Y/%m")
    SOURCE_PATH = f"{SOURCE_ROOT}/{BASE_SUBPATH}/{partition}/"
    BRONZE_PATH = f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{partition}/"
    print(f"Mode : incremental — {partition}  (last month UTC)")

print(f"Source : {SOURCE_PATH}")
print(f"Bronze : {BRONZE_PATH}")

## Cell 3 — List source JSON report files

In [ ]:
def list_files_recursive(path):
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return []
    files = []
    for item in items:
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        else:
            files.append(item)
    return files

source_files  = list_files_recursive(SOURCE_PATH)
json_files    = [f for f in source_files if f.name.endswith(".json")]

print(f"JSON files found: {len(json_files)}")
for f in json_files:
    rel = f.path.replace(SOURCE_PATH, "")
    print(f"  {rel:<50}  [{round(f.size/1024, 1)} KB]")

## Cell 4 — Copy JSON files to Bronze Volume

In [ ]:
copied  = []
skipped = []

for file_info in json_files:
    relative_path = file_info.path.replace(SOURCE_PATH, "")
    dest_path     = BRONZE_PATH + relative_path
    try:
        dbutils.fs.cp(file_info.path, dest_path)
        copied.append(dest_path)
        print(f"  COPIED  {relative_path}")
    except Exception as e:
        skipped.append((file_info.path, str(e)))
        print(f"  FAILED  {relative_path} — {e}")

print(f"\nResult: {len(copied)} copied, {len(skipped)} failed")
if skipped:
    raise Exception(f"{len(skipped)} file(s) failed — check output above.")

## Cell 5 — Verify files in Bronze Volume

In [ ]:
bronze_files = list_files_recursive(BRONZE_PATH)
bronze_json  = [f for f in bronze_files if f.name.endswith(".json")]

status = "PASS" if len(bronze_json) == len(json_files) else "FAIL"
print(f"[{status}] Source: {len(json_files)} files  →  Bronze: {len(bronze_json)} files")
for f in bronze_json:
    rel = f.path.replace(BRONZE_PATH, "")
    print(f"  {rel}")

assert len(bronze_json) == len(json_files), (
    f"Count mismatch — source: {len(json_files)}, bronze: {len(bronze_json)}"
)
print("\nVerification passed.")

## Cell 6 — Read and preview each report JSON

Each JSON file is a monthly aggregated report. Reading them confirms the files are valid and shows the top-level keys (schema varies per report type).

In [ ]:
for f in bronze_json:
    rel = f.path.replace(BRONZE_PATH, "")
    print(f"\n--- {rel} ---")
    try:
        df = spark.read.option("multiLine", True).json(f.path)
        print(f"  Top-level keys: {df.columns}")
        df.printSchema()
    except Exception as e:
        print(f"  Could not parse as JSON: {e}")
        print(f"  First 300 bytes: {dbutils.fs.head(f.path, 300)}")

## Cell 7 — Summary

In [ ]:
print("=" * 60)
print("BRONZE REPORTS JSON MIGRATION — RUN SUMMARY")
print("=" * 60)
print(f"load_type         : {load_type}")
if load_type == "incremental" and partition:
    print(f"Month (UTC-1m)    : {partition}")
print(f"Source path       : {SOURCE_PATH}")
print(f"Bronze path       : {BRONZE_PATH}")
print(f"JSON files copied : {len(copied)}")
print(f"JSON files failed : {len(skipped)}")
for f in bronze_json:
    print(f"  {f.name}")
print("=" * 60)
print("Next step: Silver layer reads these JSONs, flattens nested fields, and writes Delta tables.")